# Unified vs Segment-Specific Comparison (One-Week-Ahead Forecast)

This notebook creates a compact comparison table between:
- the **segment-specific best model** for each market category, and
- the **unified model baseline** from the unified modelling pipeline,

for the same **one-week-ahead forecasting target**.

The final table includes:
1. Market Category
2. Optimal Algorithm
3. Coefficient of Determination ($R^2$)
4. RMSE (Segment-Specific vs. Unified)
5. Error Reduction (%)

## 1. Imports and File Resolution

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

FORECAST_HORIZON = 't_plus_1_week'

def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() or ((p / 'data').exists() and (p / 'reports').exists()):
            return p
        p = p.parent
    return Path.cwd().resolve()

repo_root = find_repo_root()
tables_dir = repo_root / 'reports' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)

segment_best_path = tables_dir / 'segmentwise_best_model_per_segment.csv'
if not segment_best_path.exists():
    raise FileNotFoundError(f'Missing required file: {segment_best_path}')

# Prefer with_supply_demand outputs when available, else fallback to no_supply_demand.
unified_segment_candidates = [
    #tables_dir / 'unified_segment_evaluation_with_supply_demand.csv',
    tables_dir / 'unified_segment_evaluation_no_supply_demand.csv',
]
unified_summary_candidates = [
    #tables_dir / 'unified_cv_summary_with_supply_demand.csv',
    tables_dir / 'unified_cv_summary_no_supply_demand.csv',
]

unified_segment_path = next((p for p in unified_segment_candidates if p.exists()), None)
unified_summary_path = next((p for p in unified_summary_candidates if p.exists()), None)

if unified_segment_path is None:
    raise FileNotFoundError('Missing unified segment evaluation CSV (with/no supply-demand).')
if unified_summary_path is None:
    raise FileNotFoundError('Missing unified CV summary CSV (with/no supply-demand).')

# Safety guard: comparison table must use out-of-fold unified segment metrics.
_unified_segment_probe = pd.read_csv(unified_segment_path, nrows=50)
if 'Eval_Scope' in _unified_segment_probe.columns:
    scopes = set(_unified_segment_probe['Eval_Scope'].dropna().astype(str).unique())
    if not any('OOF' in s for s in scopes):
        raise ValueError(
            f'Unfair comparison risk: {unified_segment_path.name} is not OOF-evaluated. Re-run unified pipeline first.'
        )

if 'Forecast_Horizon' in _unified_segment_probe.columns:
    horizons = set(_unified_segment_probe['Forecast_Horizon'].dropna().astype(str).unique())
    if FORECAST_HORIZON not in horizons:
        raise ValueError(
            f'Expected forecast horizon {FORECAST_HORIZON}, but got {sorted(horizons)} in {unified_segment_path.name}'
        )

print(f'Forecast horizon: {FORECAST_HORIZON}')
print(f'Loaded segment-wise best file: {segment_best_path.name}')
print(f'Loaded unified segment file: {unified_segment_path.name}')
print(f'Loaded unified summary file: {unified_summary_path.name}')

Forecast horizon: t_plus_1_week
Loaded segment-wise best file: segmentwise_best_model_per_segment.csv
Loaded unified segment file: unified_segment_evaluation_no_supply_demand.csv
Loaded unified summary file: unified_cv_summary_no_supply_demand.csv


## 2. Build Requested Comparison Table

In [32]:
segment_best = pd.read_csv(segment_best_path).copy()
unified_seg = pd.read_csv(unified_segment_path).copy()
unified_summary = pd.read_csv(unified_summary_path).copy()

# Keep only one-week-ahead rows when horizon metadata exists.
if 'Forecast_Horizon' in segment_best.columns:
    segment_best = segment_best[segment_best['Forecast_Horizon'].astype(str) == FORECAST_HORIZON].copy()
if 'Forecast_Horizon' in unified_seg.columns:
    unified_seg = unified_seg[unified_seg['Forecast_Horizon'].astype(str) == FORECAST_HORIZON].copy()
if 'Forecast_Horizon' in unified_summary.columns:
    unified_summary = unified_summary[unified_summary['Forecast_Horizon'].astype(str) == FORECAST_HORIZON].copy()

if segment_best.empty:
    raise ValueError(f'No segment-wise rows found for forecast horizon: {FORECAST_HORIZON}')
if unified_seg.empty:
    raise ValueError(f'No unified segment rows found for forecast horizon: {FORECAST_HORIZON}')
if unified_summary.empty:
    raise ValueError(f'No unified summary rows found for forecast horizon: {FORECAST_HORIZON}')

# Normalize segment/category names for robust matching.
def norm_text(s: str) -> str:
    return str(s).strip().lower().replace('-', '_').replace(' ', '_')

segment_best['segment_norm'] = segment_best['Segment'].apply(norm_text)
unified_seg['segment_norm'] = unified_seg['Segment'].apply(norm_text)

# Harmonize unified metric column names across possible exports.
if 'RMSE' in unified_seg.columns and 'RMSE_unified' not in unified_seg.columns:
    unified_seg = unified_seg.rename(columns={'RMSE': 'RMSE_unified'})
if 'R2' in unified_seg.columns and 'R2_unified' not in unified_seg.columns:
    unified_seg = unified_seg.rename(columns={'R2': 'R2_unified'})

required_unified_cols = {'segment_norm', 'RMSE_unified', 'R2_unified'}
missing_unified = required_unified_cols - set(unified_seg.columns)
if missing_unified:
    raise ValueError(f'Unified segment file is missing required columns: {sorted(missing_unified)}')

merged = segment_best.merge(
    unified_seg[['segment_norm', 'RMSE_unified', 'R2_unified']],
    on='segment_norm',
    how='inner',
)

if merged.empty:
    raise ValueError('No matching market categories found between segment-wise and unified outputs.')

# Positive values mean segment-specific reduces error vs unified baseline.
merged['error_reduction_pct'] = np.where(
    merged['RMSE_unified'] > 0,
    (merged['RMSE_unified'] - merged['RMSE_mean']) / merged['RMSE_unified'] * 100.0,
    np.nan,
)

# Requested report table (segment-level rows).
findings_table = pd.DataFrame({
    'Market Category': merged['Segment'],
    'Optimal Algorithm': merged['Model'],
    'Coefficient of Determination (R^2)': merged['R2_mean'],
    'RMSE (Segment-Specific vs. Unified)': merged.apply(
        lambda r: f"{r['RMSE_mean']:.2f} vs {r['RMSE_unified']:.2f}", axis=1
    ),
    'Error Reduction (%)': merged['error_reduction_pct'],
    'Forecast Horizon': FORECAST_HORIZON,
})

# Add one unified baseline row as requested context.
best_unified_row = unified_summary.sort_values('RMSE_mean', ascending=True).iloc[0]
findings_table = pd.concat([
    findings_table,
    pd.DataFrame([{
        'Market Category': 'Unified',
        'Optimal Algorithm': best_unified_row['Model'],
        'Coefficient of Determination (R^2)': best_unified_row['R2_mean'],
        'RMSE (Segment-Specific vs. Unified)': f"{best_unified_row['RMSE_mean']:.2f} vs {best_unified_row['RMSE_mean']:.2f}",
        'Error Reduction (%)': 0.0,
        'Forecast Horizon': FORECAST_HORIZON,
    }]),
], ignore_index=True)

display(findings_table.round({'Coefficient of Determination (R^2)': 4, 'Error Reduction (%)': 2}))

,Market Category,Optimal Algorithm,Coefficient of Determination (R^2),RMSE (Segment-Specific vs. Unified),Error Reduction (%),Forecast Horizon
0,Dust,Random Forest,0.0856,197.21 vs 190.67,-3.43,t_plus_1_week
1,High Grown,LightGBM,-0.0317,269.31 vs 262.81,-2.47,t_plus_1_week
2,Low Grown,Random Forest,0.3125,622.59 vs 685.75,9.21,t_plus_1_week
3,Off-Grade,LightGBM,-0.0802,176.13 vs 175.63,-0.29,t_plus_1_week
4,Unified,Random Forest,0.3615,482.66 vs 482.66,0.00,t_plus_1_week


## 3. Save Table Outputs

In [33]:
suffix = '_with_supply_demand' if 'with_supply_demand' in unified_segment_path.name else '_no_supply_demand'

out_csv = tables_dir / f'unified_vs_segment_key_findings_table{suffix}.csv'
out_md = tables_dir / f'unified_vs_segment_key_findings_table{suffix}.md'

findings_table.to_csv(out_csv, index=False)

def df_to_markdown_pipe(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    lines = [
        '| ' + ' | '.join(cols) + ' |',
        '| ' + ' | '.join(['---'] * len(cols)) + ' |',
    ]
    for _, row in df.iterrows():
        vals = []
        for c in cols:
            v = row[c]
            if pd.isna(v):
                vals.append('')
            else:
                vals.append(str(v).replace('|', '\\|'))
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines)

with open(out_md, 'w', encoding='utf-8') as f:
    f.write(df_to_markdown_pipe(findings_table))

print('Saved outputs:')
print(f'- {out_csv}')
print(f'- {out_md}')

Saved outputs:
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_table_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_table_no_supply_demand.md


## 4. Statistical Validation (Fold-Wise, One-Week-Ahead)

This section reproduces the rigorous unified-vs-segment evaluation on the base processed dataset using the same one-week-ahead forecast target with:
- `TimeSeriesSplit` (5 folds)
- paired t-test on fold RMSE
- Diebold-Mariano test on forecast errors

In [34]:
import warnings
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

TARGET_BASE_COL = 'price_mid_lkr'
TARGET_COL = 'price_mid_lkr_t_plus_1w'
SEGMENT_COL = 'category_type'
SALE_NO_COL = 'sale_number'
SALE_ID_COL = 'sale_id'
N_CV_FOLDS = 5
RANDOM_STATE = 42

def make_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('regressor', RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

if 'with_supply_demand' in unified_segment_path.name:
    stat_candidates = [
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_t13_supply_demand.csv',
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_v2.csv',
    ]
else:
    stat_candidates = [
        repo_root / 'data' / 'Processed' / 'tea_preprocessed_v2.csv',
        repo_root / 'data' / 'Processed' / 'tea_preprocessed.csv',
    ]

stat_data_path = next((p for p in stat_candidates if p.exists()), None)
if stat_data_path is None:
    raise FileNotFoundError('No processed dataset found for statistical validation.')

df_raw = pd.read_csv(stat_data_path)
df = df_raw.copy()
df = df.loc[df[TARGET_BASE_COL].notna() & df[SEGMENT_COL].notna()].copy()

if 'sale_year' in df.columns:
    df['_sale_year_num'] = pd.to_numeric(df['sale_year'], errors='coerce').fillna(0)
else:
    df['_sale_year_num'] = 0

df['_sale_num'] = pd.to_numeric(df[SALE_NO_COL], errors='coerce').fillna(0)
df['_sale_id_norm'] = pd.to_numeric(df.get(SALE_ID_COL, np.nan), errors='coerce').fillna(0)
df = df.sort_values(['_sale_year_num', '_sale_num', '_sale_id_norm'], kind='mergesort').reset_index(drop=True)

# One-week-ahead target within each segment stream.
df[TARGET_COL] = df.groupby(SEGMENT_COL, sort=False)[TARGET_BASE_COL].shift(-1)
df = df.loc[df[TARGET_COL].notna()].copy()
segments = sorted(df[SEGMENT_COL].dropna().astype(str).unique())

df['_time_order'] = np.arange(len(df))

EXCLUDE_COLS = {
    SALE_ID_COL, 'sale_date_raw', 'sale_month', 'grade', 'tier',
    SEGMENT_COL, 'table_source', 'elevation', 'category',
    'price_lo_lkr', 'price_hi_lkr', 'price_range_lkr',
    TARGET_BASE_COL, TARGET_COL, 'price_mid_lkr_log', 'has_price_target', 'price_mid_usd',
    '_sale_year_num', '_sale_num', '_sale_id_norm', '_time_order',
    '_sale_date_parsed', '_sale_number_numeric', '_sale_order',
    '_current_supply_proxy', '_current_demand_proxy',
    '_historical_supply_avg', '_historical_demand_avg',
}

feature_cols = [
    c for c in df.columns
    if c not in EXCLUDE_COLS and pd.api.types.is_numeric_dtype(df[c])
]

X_all = df[feature_cols].values
y_all = df[TARGET_COL].values
segments_arr = df[SEGMENT_COL].astype(str).values

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, y_true)
    mape = float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)
    return {'rmse': rmse, 'mae': mae, 'mape': mape, 'r2': r2}

def diebold_mariano_test(e1: np.ndarray, e2: np.ndarray, horizon: int = 1, power: int = 2) -> tuple[float, float]:
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    if n < 3:
        return np.nan, np.nan

    d_mean = np.mean(d)
    gamma_0 = np.var(d, ddof=1)
    gamma_sum = 0.0
    for k in range(1, horizon):
        gamma_k = np.cov(d[k:], d[:-k], ddof=1)[0, 1] if n > k else 0.0
        gamma_sum += gamma_k
    var_d = (gamma_0 + 2 * gamma_sum) / n

    if var_d <= 0:
        return np.nan, np.nan

    dm_stat = d_mean / np.sqrt(var_d)
    k = ((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n) ** 0.5
    dm_stat_corrected = dm_stat * k if k > 0 else dm_stat
    p_value = 2.0 * stats.t.sf(np.abs(dm_stat_corrected), df=n - 1)
    return float(dm_stat_corrected), float(p_value)

tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)
fold_metrics_rows = []
prediction_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_train_fold, X_test_fold = X_all[train_idx], X_all[test_idx]
    y_train_fold, y_test_fold = y_all[train_idx], y_all[test_idx]
    seg_train_fold = segments_arr[train_idx]
    seg_test_fold = segments_arr[test_idx]

    unified_model = make_model()
    unified_model.fit(X_train_fold, y_train_fold)
    unified_pred = unified_model.predict(X_test_fold)

    segment_pred = np.full(unified_pred.shape, np.nan, dtype=float)
    for seg in segments:
        seg_train_mask = seg_train_fold == seg
        seg_test_mask = seg_test_fold == seg

        n_train_seg = int(seg_train_mask.sum())
        n_test_seg = int(seg_test_mask.sum())

        if n_test_seg == 0:
            continue

        if n_train_seg < 5:
            # No fallback to unified predictions: skip this segment in this fold if training data is insufficient.
            continue

        seg_model = make_model()
        seg_model.fit(X_train_fold[seg_train_mask], y_train_fold[seg_train_mask])
        segment_pred[seg_test_mask] = seg_model.predict(X_test_fold[seg_test_mask])

    for i, global_idx in enumerate(test_idx):
        actual = y_test_fold[i]
        u_pred = unified_pred[i]
        s_pred = segment_pred[i]
        seg = seg_test_fold[i]
        prediction_rows.append({
            'fold': fold_idx,
            'global_idx': int(global_idx),
            'segment': seg,
            'actual': float(actual),
            'pred_unified': float(u_pred),
            'pred_segment': float(s_pred) if not np.isnan(s_pred) else np.nan,
            'error_unified': float(actual - u_pred),
            'error_segment': float(actual - s_pred) if not np.isnan(s_pred) else np.nan,
            'Forecast_Horizon': FORECAST_HORIZON,
        })

    for seg in segments:
        seg_mask = seg_test_fold == seg
        if int(seg_mask.sum()) == 0:
            continue

        y_seg = y_test_fold[seg_mask]
        u_seg = unified_pred[seg_mask]
        s_seg = segment_pred[seg_mask]

        if np.all(np.isnan(s_seg)):
            continue

        m_uni = compute_metrics(y_seg, u_seg)
        m_seg = compute_metrics(y_seg, s_seg)

        fold_metrics_rows.append({
            'fold': fold_idx,
            'segment': seg,
            'n_test': int(seg_mask.sum()),
            'rmse_unified': m_uni['rmse'],
            'mae_unified': m_uni['mae'],
            'mape_unified': m_uni['mape'],
            'r2_unified': m_uni['r2'],
            'rmse_segment': m_seg['rmse'],
            'mae_segment': m_seg['mae'],
            'mape_segment': m_seg['mape'],
            'r2_segment': m_seg['r2'],
            'Forecast_Horizon': FORECAST_HORIZON,
        })

df_fold_metrics = pd.DataFrame(fold_metrics_rows).sort_values(['fold', 'segment']).reset_index(drop=True)
df_predictions = pd.DataFrame(prediction_rows).sort_values(['fold', 'global_idx']).reset_index(drop=True)

summary_rows = []
for seg, seg_folds in df_fold_metrics.groupby('segment'):
    rmse_u = seg_folds['rmse_unified'].values
    rmse_s = seg_folds['rmse_segment'].values
    r2_u = seg_folds['r2_unified'].values
    r2_s = seg_folds['r2_segment'].values

    min_len = min(len(rmse_u), len(rmse_s))
    if min_len < 2:
        t_pval = np.nan
    else:
        t_pval = float(stats.ttest_rel(rmse_u[:min_len], rmse_s[:min_len], nan_policy='omit').pvalue)

    seg_preds = df_predictions[df_predictions['segment'] == seg].copy()
    e_uni = seg_preds['error_unified'].dropna().values
    e_seg = seg_preds['error_segment'].dropna().values
    min_len = min(len(e_uni), len(e_seg))
    if min_len >= 3:
        _, dm_pval = diebold_mariano_test(e_uni[:min_len], e_seg[:min_len], horizon=1, power=2)
    else:
        dm_pval = np.nan

    rmse_u_mean = float(np.nanmean(rmse_u))
    rmse_s_mean = float(np.nanmean(rmse_s))
    pct_imp_rmse = ((rmse_u_mean - rmse_s_mean) / rmse_u_mean * 100.0) if rmse_u_mean > 0 else np.nan

    better = 'segment' if rmse_s_mean < rmse_u_mean else 'unified'

    summary_rows.append({
        'segment': seg,
        'n_folds': int(seg_folds.shape[0]),
        'n_obs': int(seg_preds.shape[0]),
        'rmse_unified_mean': rmse_u_mean,
        'rmse_segment_mean': rmse_s_mean,
        'pct_improvement_rmse': pct_imp_rmse,
        'r2_unified_mean': float(np.nanmean(r2_u)),
        'r2_segment_mean': float(np.nanmean(r2_s)),
        'paired_t_pvalue': t_pval,
        'dm_pvalue': dm_pval,
        'better_model': better,
        'Forecast_Horizon': FORECAST_HORIZON,
    })

df_stat_summary = pd.DataFrame(summary_rows).sort_values('segment').reset_index(drop=True)
display(df_stat_summary.round(4))
print(f"Statistical validation source data: {stat_data_path.name}")
print(f"Features used: {len(feature_cols)}")
print(f"Forecast horizon: {FORECAST_HORIZON}")

,segment,n_folds,n_obs,rmse_unified_mean,rmse_segment_mean,pct_improvement_rmse,r2_unified_mean,r2_segment_mean,paired_t_pvalue,dm_pvalue,better_model,Forecast_Horizon
0,dust,5,995,196.3434,201.6600,-2.7078,0.0772,0.0124,0.3964,0.0001,unified,t_plus_1_week
1,high_grown,5,1095,248.5689,251.4852,-1.1732,0.0737,0.0444,0.4999,0.0752,unified,t_plus_1_week
2,low_grown,5,2568,716.1565,714.7008,0.2033,0.0414,0.0455,0.1466,0.2077,segment,t_plus_1_week
3,off_grade,5,987,174.5932,176.2075,-0.9246,-0.0629,-0.0850,0.2930,0.0224,unified,t_plus_1_week


Statistical validation source data: tea_preprocessed_v2.csv
Features used: 231
Forecast horizon: t_plus_1_week


## 5. Save Statistical Validation Outputs

In [35]:
stat_fold_csv = tables_dir / f'unified_vs_segment_key_findings_fold_metrics{suffix}.csv'
stat_pred_csv = tables_dir / f'unified_vs_segment_key_findings_predictions{suffix}.csv'
stat_summary_csv = tables_dir / f'unified_vs_segment_key_findings_stat_summary{suffix}.csv'

df_fold_metrics.to_csv(stat_fold_csv, index=False)
df_predictions.to_csv(stat_pred_csv, index=False)
df_stat_summary.to_csv(stat_summary_csv, index=False)

print('Saved statistical outputs:')
print(f'- {stat_fold_csv}')
print(f'- {stat_pred_csv}')
print(f'- {stat_summary_csv}')

Saved statistical outputs:
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_fold_metrics_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_predictions_no_supply_demand.csv
- C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\reports\tables\unified_vs_segment_key_findings_stat_summary_no_supply_demand.csv
